# AML Investigation Prioritization — Model Development

**Decision question:** At a fixed investigation capacity, how many positive *synthetic benchmark labels* can the risk-ranked queue capture?

[Live interactive case study](https://mohammedmia1.github.io/AML_Data_Science_Project/) · [Project guide](../PROJECT_GUIDE.md) · [Recorded metrics](../reports/metrics.json)

![Capacity-constrained AML synthetic account-day evaluation](../images/AML_Canonical_Dashboard.png)

### Published held-out results (recorded canonical run)

| Scored account-days | Reviews | Positive labels selected | Precision | Recall | Lift vs random |
|--:|--:|--:|--:|--:|--:|
| **494,707** | **2,747** | **389** | **14.16%** | **25.95%** | **46.73×** |

The figures above are **recorded results from the independently reproduced canonical pipeline**, not outputs generated by opening this notebook. Code cells below are deliberately **unexecuted in the published file**; full execution requires the exact source TSV (see [data contract](../data/README.md)).

**Notebook roadmap:** (1) inspect the grain and dates, (2) prepare nine leak-aware features, (3) fit a logistic baseline and the final XGBoost model, (4) evaluate a fixed-capacity queue, and (5) reproduce the authoritative artifacts. No real-world criminal detection or investigator impact is claimed.


## 1. Data contract and temporal holdout

Each observation is one account-day. The data has no personal customer records. Training uses **September 1–6, 2022**, validation uses **September 7–8**, and the untouched test uses **September 9–10**.

We use only training labels to set class weights and do not include `account_id`, `activity_date`, or the target among model features. The validation queue is frozen before test evaluation.

In [ ]:
from pathlib import Path
import os
import pandas as pd

# Works when Jupyter starts in either the repository root or notebooks/.
repo_root = Path.cwd()
if not (repo_root / "src" / "run_canonical.py").exists():
    repo_root = repo_root.parent
if not (repo_root / "src" / "run_canonical.py").exists():
    raise FileNotFoundError("Launch Jupyter from the project root or notebooks/ directory")

input_dir = Path(os.environ.get("AML_INPUT_DIR", str(repo_root / "data" / "local")))
df = pd.read_csv(input_dir / "aml_modeling_dev.tsv", sep="\t")
df["activity_date"] = pd.to_datetime(df["activity_date"])
assert not df[["account_id", "activity_date"]].duplicated().any()
print("Rows:", len(df), "Dates:", df.activity_date.min(), "to", df.activity_date.max())
print("Positive label rate:", df.has_laundering_tx.mean())


## 2. Features and leakage control

The model uses nine predictors. Six skewed measures receive `log1p`; three ratios/diversity features remain untransformed. Missing/nonfinite features cause a validation error rather than silently entering the model. The date is used to split observations, not as a predictive input.

In [ ]:
import numpy as np
log_features = ["transaction_count", "total_sent", "total_received", "avg_transaction_amount", "max_transaction_amount", "unique_banks"]
raw_features = ["outgoing_ratio", "currency_diversity", "rapid_transaction_ratio"]
def make_features(data):
    x = pd.DataFrame(index=data.index)
    for name in log_features:
        x["log_" + name] = np.log1p(data[name])
    for name in raw_features:
        x[name] = data[name]
    assert np.isfinite(x.to_numpy()).all()
    return x
train = df[df.activity_date.between("2022-09-01", "2022-09-06")]
validation = df[df.activity_date.between("2022-09-07", "2022-09-08")]
test = df[df.activity_date.between("2022-09-09", "2022-09-10")]
X_train, X_validation, X_test = [make_features(d) for d in (train, validation, test)]
print("Train / validation / test:", len(train), len(validation), len(test))

## 3. Baseline and final ranking model

**Why two approaches?** Logistic regression provides a simpler, interpretable development comparator; XGBoost models nonlinear relationships across nine engineered activity features. Both models are evaluated on the *validation* period. The published final model uses XGBoost with fixed hyperparameters and training-derived class weights.

**Evidence boundary:** This repository does not publish numeric baseline-versus-XGBoost gains. Do not infer a quantified improvement from the existence of two code paths. The final held-out metrics belong to the canonical XGBoost run only.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
y_train = train["has_laundering_tx"]
y_validation = validation["has_laundering_tx"]
baseline = Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))])
baseline.fit(X_train, y_train)
print("Logistic validation PR-AUC:", round(average_precision_score(y_validation, baseline.predict_proba(X_validation)[:, 1]), 4))

In [ ]:
from xgboost import XGBClassifier
weight = (y_train.eq(0).sum() / y_train.eq(1).sum())
model = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, objective="binary:logistic", eval_metric="aucpr", scale_pos_weight=weight, random_state=42, n_jobs=2)
model.fit(X_train, y_train)
print("XGBoost validation PR-AUC:", round(average_precision_score(y_validation, model.predict_proba(X_validation)[:, 1]), 4))

## 4. Freeze review capacity before testing

Investigation resources constrain the *number of rows reviewed*, so ranking quality at a fixed queue size is more useful here than accuracy at an arbitrary probability threshold. Choose a **2,500-row** validation workload, carry the same fraction into the held-out test set (**2,747 rows**), and select the highest-risk scores with a stable tie break.

**Metrics:** `precision = selected positives / reviewed rows`; `recall = selected positives / all test positives`; `lift = precision / overall test prevalence`. ROC-AUC and average precision summarize discrimination across thresholds. No test-period result is used to choose review capacity.


In [ ]:
from sklearn.metrics import roc_auc_score
y_test = test["has_laundering_tx"].to_numpy()
prob = model.predict_proba(X_test)[:, 1]
capacity = round(len(test) * 2500 / len(validation))
order = np.argsort(-prob, kind="stable")
tp = int(y_test[order[:capacity]].sum())
precision, recall = tp / capacity, tp / int(y_test.sum())
lift = precision / float(y_test.mean())
print({"account_days": len(test), "review_capacity": capacity, "selected_positives": tp, "precision": precision, "recall": recall, "lift": lift, "ROC_AUC": roc_auc_score(y_test, prob), "PR_AUC": average_precision_score(y_test, prob)})

## 5. Export the authoritative reproducible artifacts

Run the canonical script instead of maintaining a second, diverging export implementation in the notebook. With the **exact checksum-verified TSV**, it saves the model, 494,707-row scored export, exact metric JSON and 15,000-point capacity curve to ignored local `artifacts/`.

This published notebook intentionally has **no stored execution outputs**; its role is the inspectable analysis recipe. Run `python tests/check_artifacts.py` at repository root to verify the committed results without downloading the input TSV. GitHub Actions does **not** retrain XGBoost.


In [ ]:
import subprocess
import sys

# Re-run the authoritative CLI implementation instead of duplicating export logic.
# AML_INPUT_DIR may point to the frozen TSV; AML_OUTPUT_DIR defaults to ignored artifacts/.
subprocess.run(
    [sys.executable, str(repo_root / "src" / "run_canonical.py")],
    check=True,
    cwd=repo_root,
    env={**os.environ, "AML_INPUT_DIR": str(input_dir)},
)


## 6. Interpretation and limitations

At **2,747** selected account-days, the held-out run finds **389** positive benchmark labels among **1,499** test positives (**14.16%** precision, **25.95%** recall and **46.73×** lift). ROC-AUC is **0.8523** and average precision is **0.1214**.

**This demonstrates ranking concentration under a constrained hypothetical review policy—not confirmed financial crime detection.** Synthetic labels, a two-day test period, uncalibrated scores, and unknown investigative costs limit external validity. A raw-transaction SQL rebuild has not been reconciled to the frozen TSV.

For the complete technical argument, see [MODEL_EVALUATION.md](../reports/MODEL_EVALUATION.md) and the [project walkthrough](../PROJECT_GUIDE.md).
